In [1]:
import pandas as pd
import random

# 1. Configuration
NUM_SAMPLES = 5000

# The Latent Nodes (Root Causes)
# We include "Healthy" as the baseline state.
faults = ['Healthy', 'Compute_Overload', 'Memory_Leak', 'Network_Partition', 'App_Crash']

# We want the server to be healthy most of the time to reflect reality
# Weights: 60% Healthy, 10% for each specific fault
fault_weights = [0.60, 0.10, 0.10, 0.10, 0.10]

data = []

# 2. Generate the data based on your Causal Edges
for _ in range(NUM_SAMPLES):
    # Pick the true state of the server for this moment in time
    true_state = random.choices(faults, weights=fault_weights)[0]
    
    # Initialize baseline observable metrics
    cpu = 'Normal'
    ram = 'Normal'
    latency = 'Normal'
    error = 'Zero'
    
    # Apply the causal effects based on the chosen root cause
    if true_state == 'Healthy':
        # Add a tiny bit of background noise (e.g., normal garbage collection spikes)
        if random.random() < 0.05: cpu = 'High'
        if random.random() < 0.05: ram = 'High'
        if random.random() < 0.05: latency = 'Elevated'
            
    elif true_state == 'Compute_Overload':
        # Causes -> CPU_Usage (High/Critical) & API_Latency (Elevated)
        cpu = random.choices(['High', 'Critical'], weights=[0.3, 0.7])[0]
        latency = random.choices(['Elevated', 'Timeout'], weights=[0.8, 0.2])[0]
        
    elif true_state == 'Memory_Leak':
        # Causes -> RAM_Usage (High/Critical) & API_Latency (Elevated)
        ram = random.choices(['High', 'Critical'], weights=[0.2, 0.8])[0]
        latency = random.choices(['Elevated', 'Timeout'], weights=[0.8, 0.2])[0]
        
    elif true_state == 'Network_Partition':
        # Causes -> API_Latency (Timeout) & Error_Rate (Spiking)
        latency = 'Timeout'
        # Errors might not spike instantly, so we add a little probability variance
        error = random.choices(['Zero', 'Spiking'], weights=[0.2, 0.8])[0]
        
    elif true_state == 'App_Crash':
        # Causes -> Error_Rate (Spiking)
        error = 'Spiking'
        # App might hang slightly before throwing the 500 error
        if random.random() < 0.3: latency = 'Elevated'

    # 3. Store the record
    data.append({
        'Root_Cause': true_state,
        'CPU_Usage': cpu,
        'RAM_Usage': ram,
        'API_Latency': latency,
        'Error_Rate': error
    })

# 4. Export and Verify
df = pd.DataFrame(data)

df

,Root_Cause,CPU_Usage,RAM_Usage,API_Latency,Error_Rate
0,Healthy,Normal,Normal,Normal,Zero
1,Healthy,Normal,Normal,Normal,Zero
2,App_Crash,Normal,Normal,Normal,Spiking
3,Healthy,Normal,Normal,Normal,Zero
4,Healthy,Normal,Normal,Normal,Zero
...,...,...,...,...,...
4995,Network_Partition,Normal,Normal,Timeout,Spiking
4996,Healthy,Normal,Normal,Normal,Zero
4997,Healthy,Normal,Normal,Normal,Zero
4998,Healthy,Normal,Normal,Normal,Zero


In [2]:
# Save to CSV so you can load it into your MLflow pipeline later
df.to_csv('simulated_server_telemetry.csv', index=False)

print(f"Successfully generated {len(df)} rows of telemetry data.\n")
print("Distribution of Root Causes:")
print(df['Root_Cause'].value_counts(normalize=True) * 100)

print("\nSample Data:")
print(df.sample(5))

Successfully generated 5000 rows of telemetry data.

Distribution of Root Causes:
Root_Cause
Healthy              58.68
Compute_Overload     10.90
Memory_Leak          10.56
Network_Partition    10.10
App_Crash             9.76
Name: proportion, dtype: float64

Sample Data:
             Root_Cause CPU_Usage RAM_Usage API_Latency Error_Rate
2271          App_Crash    Normal    Normal    Elevated    Spiking
754             Healthy    Normal    Normal      Normal       Zero
2938   Compute_Overload  Critical    Normal    Elevated       Zero
2134  Network_Partition    Normal    Normal     Timeout    Spiking
2408            Healthy    Normal    Normal      Normal       Zero
